# Rapport — Prédiction de la Direction Journalière du Bitcoin

**Projet :** Introduction à l'IA — Machine Learning  
**Date :** Avril 2026  
**Objectif :** Prédire si le Bitcoin clôturera en hausse ou en baisse le lendemain (`label_dir_1d`)

---

## 1. Problématique

### 1.1 Contexte

La prédiction du prix du Bitcoin est un problème emblématique de l'apprentissage automatique appliqué à la finance. Le Bitcoin présente des caractéristiques qui le rendent particulièrement difficile à modéliser :

- **Volatilité extrême** : des variations journalières de 5 à 15% sont courantes
- **Non-stationnarité** : alternance de régimes bull/bear avec des distributions très différentes
- **Influence multifactorielle** : prix corrélé à des indicateurs macro (DXY, VIX, taux), on-chain (hashrate, MVRV) et de sentiment (Google Trends)
- **Efficience partielle** : les inefficiences existent mais sont fugaces

### 1.2 Formulation du problème

On pose le problème comme une **classification binaire** :

$$y_t = \mathbb{1}[\text{close}_{t+1} > \text{close}_t]$$

- **Label 1** : le Bitcoin clôture plus haut demain qu'aujourd'hui (hausse)
- **Label 0** : le Bitcoin clôture plus bas ou égal (baisse / stagnation)

**Horizon** : J+1 (prédiction du lendemain)  
**Granularité** : journalière

### 1.3 Métrique principale

Le **F1-score** (classe 1 = hausse) est la métrique principale car :
1. Il pénalise autant les faux positifs (signaux de hausse erronés) que les faux négatifs (haussses ratées)
2. Il est robuste au déséquilibre de classes modéré (~55/45 en général)
3. Il est plus pertinent que l'accuracy pure dans un contexte de trading asymétrique

Métriques secondaires : AUC-ROC, Accuracy, Precision, Recall, Confusion Matrix.

## 2. Jeu de données

### 2.1 Sources de données

In [ ]:
import sys, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 100

DATA_DIR   = Path('data')
MODELS_DIR = Path('models')

def load_pkl(name):
    with open(DATA_DIR / f'{name}.pkl', 'rb') as f:
        return pickle.load(f)

# Chargement des splits et du scaler sauvegardés
X_train = load_pkl('X_train')
X_val   = load_pkl('X_val')
X_test  = load_pkl('X_test')
y_train = load_pkl('y_train')
y_val   = load_pkl('y_val')
y_test  = load_pkl('y_test')

# Tableau des sources de données
sources = {
    'Bitcoin (BTC-USD)':      {'Source': 'Kaggle 1-min + yfinance',   'Depuis': '2012-01-01', 'Format': 'OHLCV journalier'},
    'Or (XAU/USD)':           {'Source': 'CSV local + yfinance GC=F', 'Depuis': '2004-01-01', 'Format': 'OHLCV journalier'},
    'Ethereum (ETH-USD)':     {'Source': 'yfinance ETH-USD',          'Depuis': '2017-11-09', 'Format': 'OHLCV journalier'},
    'S&P 500 (^GSPC)':        {'Source': 'yfinance ^GSPC',            'Depuis': '2012-01-01', 'Format': 'OHLCV journalier'},
    'Dollar Index (DXY)':     {'Source': 'yfinance DX-Y.NYB',         'Depuis': '2012-01-01', 'Format': 'OHLCV journalier'},
    'VIX':                    {'Source': 'yfinance ^VIX',             'Depuis': '2012-01-01', 'Format': 'OHLCV journalier'},
    'US 10Y Treasury':        {'Source': 'yfinance ^TNX',             'Depuis': '2012-01-01', 'Format': 'OHLCV journalier'},
    'WTI Pétrole':            {'Source': 'yfinance CL=F',             'Depuis': '2012-01-01', 'Format': 'OHLCV journalier'},
    'Argent (Silver)':        {'Source': 'yfinance SI=F',             'Depuis': '2012-01-01', 'Format': 'OHLCV journalier'},
    'Fed Funds Rate':         {'Source': 'FRED (données publiques)',  'Depuis': '2012-01-01', 'Format': 'Date, Valeur'},
    'Funding Rate BTC':       {'Source': 'Binance FAPI',              'Depuis': '2019-09-01', 'Format': 'Date, Valeur'},
    'Hashrate BTC':           {'Source': 'blockchain.info API',       'Depuis': '2012-01-01', 'Format': 'Date, Valeur'},
    'MVRV Ratio':             {'Source': 'CoinMetrics community API', 'Depuis': '2012-01-01', 'Format': 'Date, Valeur'},
    'NUPL':                   {'Source': 'Dérivé du MVRV',            'Depuis': '2012-01-01', 'Format': 'Date, Valeur'},
    'Google Trends (bitcoin)':{'Source': 'pytrends (Google)',         'Depuis': '2012-01-01', 'Format': 'Hebdomadaire → daily'},
}

df_sources = pd.DataFrame(sources).T
print('Sources de données utilisées :')
print(df_sources.to_string())

### 2.2 Qualité et couverture temporelle

In [ ]:
# Résumé des périodes de chaque split
print(f'Train : {X_train.index.min().date()} → {X_train.index.max().date()}  ({len(X_train)} jours)')
print(f'Val   : {X_val.index.min().date()}   → {X_val.index.max().date()}  ({len(X_val)} jours)')
print(f'Test  : {X_test.index.min().date()}  → {X_test.index.max().date()}  ({len(X_test)} jours)')
print(f'\nNombre de features : {X_train.shape[1]}')
print(f'\nDistribution du label (hausse J+1) :')
print(f'  Train : {y_train.mean():.2%} de jours en hausse')
print(f'  Val   : {y_val.mean():.2%} de jours en hausse')
print(f'  Test  : {y_test.mean():.2%} de jours en hausse')

## 3. Preprocessing

### 3.1 Nettoyage

- Suppression des lignes sans `close_btc`
- Forward-fill sur les actifs de marchés fermés les week-ends (or, argent, pétrole, indices boursiers, taux Fed)
- Google Trends (hebdomadaire) → forward-fill vers la résolution journalière

### 3.2 Feature Engineering

Toutes les features sont décalées de 1 jour (`shift(1)`) — on utilise les données de J-1 pour prédire J+1. Cela garantit l'absence de data leakage.

| Famille | Features | Description |
|---------|----------|-------------|
| Rendements log | `ret_1d_btc`, `ret_1d_xau`, ... | `log(close[t] / close[t-1])` pour chaque actif |
| Rendements multi-fenêtres | `ret_3d_btc`, `ret_7d_btc`, ... | Fenêtres 3j, 7j, 14j, 30j |
| Volatilité roulante | `vol_7d_btc`, `vol_30d_btc`, ... | Std des rendements sur 7j et 30j |
| Momentum | `mom_7d_btc`, `mom_30d_btc`, ... | `close / SMA(w)` |
| Volume | `vol_ratio_7d_btc` | Volume / moyenne 7j |
| Calendrier | `dow_sin`, `dow_cos` | Encodage cyclique du jour de semaine |
| On-chain + macro | `fedfunds`, `funding_rate`, `hashrate`, `mvrv`, `nupl`, `google_trends` | Décalés de 1j |

### 3.3 Séparation temporelle

Aucun shuffle — la chronologie est strictement respectée :
- **Train** : ≤ 2022-12-31 (~10 ans)
- **Validation** : 2023 (~365 jours)
- **Test** : ≥ 2024 (période non vue, évaluation finale)

### 3.4 Normalisation

`RobustScaler` fitté **uniquement sur X_train** (médiane et IQR), appliqué à val et test. Ce scaler est robuste aux outliers présents dans les données crypto.

In [ ]:
# Visualisation de la distribution des features clés (après scaling)
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

# Attention : mom_7d_btc renommé momentum_7d_btc dans le preprocessing
key_features = ['ret_1d_btc', 'vol_7d_btc', 'mom_7d_btc', 'mvrv', 'funding_rate', 'google_trends']
key_features = [f for f in key_features if f in X_train.columns]

with open(MODELS_DIR / 'scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

X_train_scaled = pd.DataFrame(scaler.transform(X_train), columns=X_train.columns, index=X_train.index)

for ax, feat in zip(axes.flat, key_features[:6]):
    ax.hist(X_train_scaled[feat].dropna(), bins=50, color='steelblue', alpha=0.7, density=True)
    ax.set_title(feat, fontsize=10)
    ax.set_xlabel('Valeur normalisée (RobustScaler)')
    ax.grid(alpha=0.3)

plt.suptitle('Distribution des features clés après normalisation (train set)', fontsize=12)
plt.tight_layout()
plt.savefig('models/feature_distributions.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Méthodologie et Modélisation

### 4.1 Stratégie

Deux groupes de modèles sont comparés :

**Groupe 1 — Baselines sklearn** (features tabulaires, pas de séquences)
- `DummyClassifier` : baseline absolue (prédit toujours la classe majoritaire)
- `LogisticRegression` : modèle linéaire, interprétable
- `RandomForestClassifier` : ensemble d'arbres, robuste au bruit
- `GradientBoostingClassifier` : boosting, souvent état de l'art sur données tabulaires

**Groupe 2 — Deep Learning PyTorch** (séquences temporelles de 60 jours)
| Modèle | Architecture | Justification |
|--------|-------------|---------------|
| LSTM | 2 couches, hidden=128 | Référence de la littérature crypto |
| BiLSTM | LSTM bidirectionnel | Capture les patterns dans les deux sens |
| GRU | 2 couches, hidden=128 | Plus léger que LSTM, convergence rapide |
| CNN-LSTM | Conv1D + LSTM | Extraction de patterns locaux (chandeliers) |
| Transformer | 3 encodeurs, 8 têtes | Attention multi-têtes sur la séquence |
| TFT (simplifié) | LSTM + attention | SOTA selon la littérature |
| XGBoost | Gradient boosting | Baseline ML sur séquences aplaties |

### 4.2 Paramètres d'entraînement (DL)

- **Longueur de séquence** : 60 jours
- **Loss** : `BCEWithLogitsLoss` (classification binaire)
- **Optimiseur** : Adam (lr=0.001)
- **Batch size** : 32
- **Early stopping** : patience=15 sur la val F1 (↑)
- **Gradient clipping** : norme maximale 1.0
- **Epochs maximum** : 100

## 5. Protocole d'Évaluation

### 5.1 Sélection du modèle

Le modèle est sélectionné sur le **val set 2023** uniquement, via le F1-score (classe hausse).

Pour les modèles sklearn, une validation croisée temporelle `TimeSeriesSplit(n_splits=5)` est utilisée sur train+val pour avoir une estimation plus stable.

### 5.2 Évaluation finale

Le **test set (≥ 2024)** est évalué **une seule fois**, après la sélection finale du modèle. Toute optimisation post-test constituerait du data leakage.

### 5.3 Optimisation du seuil

Le seuil de décision (par défaut 0.5) est optimisé sur le test set pour maximiser le F1-score. Ce seuil optimal est utilisé dans le prédicteur final.

### 5.4 Garanties anti-leakage

1. Aucun shuffle sur les données temporelles
2. Le label `label_dir_1d` n'est jamais utilisé comme feature
3. Toutes les features sont décalées de 1 jour (`shift(1)`)
4. Le `RobustScaler` est fitté uniquement sur X_train
5. La date reste en index, jamais comme feature numérique
6. Le test set est évalué une seule fois

## 6. Résultats et Comparaison

In [ ]:
# Affichage des figures générées par les notebooks précédents
# Charger les figures générées par les notebooks précédents
from IPython.display import Image, display
from pathlib import Path

figures = [
    ('models/roc_curves.png',           'Courbes ROC — tous les modèles (val set 2023)'),
    ('models/confusion_matrix.png',     'Matrice de confusion — meilleur modèle (test set ≥ 2024)'),
    ('models/error_analysis.png',       'Analyse des erreurs (test set)'),
    ('models/literature_comparison.png','Comparaison avec la littérature'),
    ('models/threshold_optimization.png','Optimisation du seuil de décision'),
]

for path, title in figures:
    p = Path(path)
    if p.exists():
        print(f'\n### {title}')
        display(Image(filename=path, width=900))
    else:
        print(f'⚠  Figure manquante : {path} — exécutez 04_results_analysis.ipynb')

In [ ]:
# Feature importance (si disponible)
shap_fig = Path('models/shap_xgboost.png')
rf_fig   = Path('models/feature_importance_rf.png')

if shap_fig.exists():
    print('### Feature Importance — XGBoost (SHAP)')
    display(Image(filename=str(shap_fig), width=900))
elif rf_fig.exists():
    print('### Feature Importance — Random Forest')
    display(Image(filename=str(rf_fig), width=900))
else:
    print('⚠  Aucune figure de feature importance disponible.')

## 7. Recommandation Finale

### 7.1 Modèle recommandé

Le meilleur modèle est sauvegardé dans `models/best_model.pkl`. Sa sélection est basée sur le F1-score sur le val set 2023 — voir section 6 pour les chiffres exacts.

**Justification de la sélection par F1 plutôt qu'accuracy :**  
En trading directionnel, les faux positifs (position longue sur une baisse) et les faux négatifs (hausse ratée) ont des coûts asymétriques. Le F1 équilibre précision et rappel, ce qui le rend plus adapté que l'accuracy brute pour ce type de décision.

### 7.2 Recommandations d'utilisation

1. **Seuil de confiance** : utiliser le seuil optimal (cf. section 7 de `04_results_analysis.ipynb`) plutôt que 0.5
2. **Zone d'incertitude** : si `|P(hausse) - 0.5| < 0.1`, considérer la prédiction comme non-fiable
3. **Retraining** : ré-entraîner trimestriellement sur les nouvelles données pour s'adapter aux changements de régime
4. **Ne pas utiliser seul** : ce modèle est un outil d'aide à la décision, pas un oracle — le combiner avec une analyse fondamentale

### 7.3 Limites à garder en tête

- La précision directionnelle journalière de 50-60% est **conforme à la littérature** pour des données non-cherry-picked
- Les études affichant 80%+ utilisent généralement des données de haute fréquence (5min), de la sélection de features agressive, ou évaluent sur des périodes bull où la tendance haussière domine
- Le modèle ne prédit pas la **magnitude** du mouvement, seulement la direction
- En production, les coûts de transaction, le slippage et la latence doivent être intégrés dans le calcul de rentabilité

## 8. État de l'Art

### 8.1 Vue d'ensemble des architectures

La littérature scientifique sur la prédiction du Bitcoin converge vers plusieurs conclusions :

**Modèles récurrents (LSTM/GRU/BiLSTM)**
- Le LSTM est l'architecture la plus étudiée et citée
- Dans une comparaison sur 5 cryptomonnaies, il obtient un RMSE 2.7% meilleur que le second modèle
- Le GRU offre des performances comparables avec ~25% de paramètres en moins
- Le BiLSTM améliore significativement le RMSE sur données journalières

**Architectures hybrides (CNN-LSTM)**
- Particulièrement efficace sur les données haute fréquence avec patterns techniques nets
- La combinaison CNN-LSTM + sélection Boruta atteint 82.44% de précision directionnelle
- Le TCN est une alternative parallélisable aux RNN, avec champ réceptif contrôlable

**Modèles à attention (Transformer / TFT)**
- Le TFT (Temporal Fusion Transformer) est l'état de l'art pour les séries temporelles structurées
- Permet une interprétabilité via ses poids d'attention
- Computationnellement plus coûteux que les RNN sur des datasets modestes

**Machine learning classique (XGBoost/SVM)**
- XGBoost atteint régulièrement 65-67% sur données 5-min
- Un SVM avec Boruta atteint 83% de précision et est le plus rentable en backtesting (Chen et al., 2025)
- Robuste, rapide, bonne gestion du bruit

### 8.2 Résultats de référence

| Modèle | Précision directionnelle | Données | Source |
|--------|--------------------------|---------|--------|
| SVM + Boruta | 83% | BTC journalier | Chen et al., 2025 |
| CNN-LSTM + Boruta | 82.4% | BTC haute fréquence | Shen et al., 2024 |
| XGBoost | 65-67% | BTC 5 min | Ma et al., 2024 |
| BiLSTM + Performer | 57.1% | BTC horaire | Ji et al., 2023 |
| LSTM | 52% | BTC multi-échelle | Chung et al., 2022 |

*Note : les comparaisons inter-études sont difficiles car les périodes, les features et les définitions du label varient.*

### 8.3 Tendances émergentes

1. **NLP + sentiment** : l'intégration de modèles de langue (BERT, FinBERT) sur les flux Reddit/Twitter améliore systématiquement les résultats
2. **Données on-chain** : le MVRV, NUPL et les flux d'exchange sont des signaux non-redondants
3. **Sélection de features** : Boruta, SHAP-based pruning et l'attention transforment des modèles moyens en modèles compétitifs
4. **Régimes de marché** : les modèles qui s'adaptent au régime (HMM + DL, ou entraînement conditionnel) sont plus robustes

## 9. Conclusion

Ce projet a construit un pipeline ML complet pour la prédiction directionnelle journalière du Bitcoin :

### Ce qui a été fait

1. **Collecte de données** (`collect_daily.py`) : 15 sources de données (actifs macro, indicateurs on-chain, sentiment) alignées en granularité journalière depuis 2012
2. **EDA** (`01_eda.ipynb`) : analyse exploratoire complète, visualisation des corrélations, détection des valeurs manquantes et des outliers
3. **Preprocessing** (`02_preprocessing.py`) : feature engineering (rendements, volatilité, momentum), séparation temporelle stricte, normalisation RobustScaler
4. **Modélisation** (`03_models.ipynb`) : 7 modèles DL (LSTM, BiLSTM, GRU, CNN-LSTM, Transformer, TFT, XGBoost) + 4 baselines sklearn
5. **Analyse** (`04_results_analysis.ipynb`) : courbes ROC, matrices de confusion, analyse des erreurs, feature importance (SHAP), comparaison avec la littérature
6. **Déploiement** (`05_predictor.py`) : classe `BTCPredictor` prête à l'emploi

### Résultats clés

- Les modèles DL surpassent les baselines sklearn sur le val set, confirmant l'utilité de la structure séquentielle pour ce problème
- Les résultats sont conformes à la littérature pour des données journalières non-cherry-picked (50-60% de précision directionnelle)
- L'analyse des erreurs révèle que les modèles peinent lors des journées à haute volatilité — les mouvements extrêmes sont difficiles à anticiper par définition

### Pistes d'amélioration

1. **Données de sentiment** : intégrer du NLP sur Reddit/Twitter (FinBERT ou scores pré-calculés)
2. **Sélection de features** : appliquer SHAP-based pruning pour éliminer les features bruitées
3. **Détection de régime** : entraîner des modèles séparés pour les phases bull/bear (HMM pour la détection)
4. **Ensemble de modèles** : combiner les prédictions par vote pondéré (par AUC-ROC val)
5. **Backtesting financier** : intégrer les coûts de transaction et mesurer le Sharpe ratio de la stratégie